# NB04 · Calibrated multi-horizon risk head (LightGBM) + conformal uncertainty + risk states  (CPU · ~30–60 min)

One **hazard-agnostic** model (hazard identity is never a feature). For each zone and each event type, `P(critical event onsets in (t, t+h])` at h ∈ {1,3,6,12,24,48,72}:

* LightGBM with **`horizon` as a feature + monotone constraint (+1)** → a valid CDF → time-to-critical by inversion. 8 zone models (7 event types + `any`) and 1 city-level **cascade** model.
* Isotonic calibration per (type, horizon) on validation episodes; monotone-in-h enforced after calibration.
* **CQR + Mondrian bins by evidence coverage + ACI** around time-to-critical intervals (outcomes fed back only when they would be known).
* 4-state mapping (NORMAL/WATCH/HIGH RISK/CRITICAL) with hysteresis, thresholds tuned to a **false-alarm budget**.
* **Metrics:** AUROC/AP/Brier (+skill vs climatology)/ECE per horizon, reliability diagrams, **lead time at fixed false-alarm rate**, interval coverage by data-quality bin, fault-profile table, **dropout sweep (graceful degradation)**, held-out city, baselines (persistence-threshold, Chronos-only, climatology), **ablations**, grouped TreeSHAP.

**Inputs:** NB00 output (required). NB02 `hawkes_prior.npz` and NB01 `chronos_feats/` are **optional but recommended** — detected automatically; ablations are fair (same episodes).
**Outputs:** models, calibrators, thresholds, `risk_predictions_*.csv.gz` (for NB05), figs.  Each model checkpointed.

In [ ]:
import subprocess, sys
try: import lightgbm
except Exception: print(subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'lightgbm'], capture_output=True, text=True).stdout[-200:])

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
QUICK = False
CFG = dict(rounds=600, es=40, abl_rows=250_000, far_budget=1.0, n_sweep_eps=40, shap_rows=20000)
if QUICK: CFG.update(rounds=30, es=10, abl_rows=20_000, n_sweep_eps=6, shap_rows=2000)
ORIG = np.arange(48, 168, 3); H = W.HORIZONS; NH = len(H); TYPES = W.CRIT_NAMES + ['any']; NT = len(TYPES)
MAN = pd.read_csv(find('w72_manifest.csv')); ROOT = os.path.dirname(find('w72_manifest.csv')); CITY = {'anchor': W.make_city('anchor'), 'heldout': W.make_city('heldout')}
hp = find('hawkes_prior.npz', False); HK = None
if hp: z = np.load(hp); HK = dict(A=z['A'], mu=z['mu'], s=z['s'], taus=z['taus'], rho=float(z['rho'])); print('Hawkes prior found: rho', round(HK['rho'], 3))
else: print('NB02 output not attached -> no Hawkes features')
CHD = os.path.dirname(os.path.dirname(find('chronos_info.json', False))) if find('chronos_info.json', False) else None
CFD = os.path.join(os.path.dirname(find('chronos_info.json')), 'chronos_feats') if find('chronos_info.json', False) else None
print('Chronos features:', CFD)
def ch_path(split, ep): return f'{CFD}/{split}/{ep}.npz' if CFD else None
USE_CH = bool(CFD) and any(glob.glob(f'{CFD}/*/*.npz')); USE_HK = HK is not None; print('USE_CHRONOS', USE_CH, '| USE_HAWKES', USE_HK)

## Dataset construction (strict as-of: features at origin t use only data ingested ≤ t; labels come from the separate label step)

In [ ]:
NEXT = lambda on: np.minimum.accumulate(np.where(on, np.arange(len(on))[:, None] * np.ones((1, on.shape[1]), int), 10**6)[::-1], axis=0)[::-1]
def build_episode(r, city, drop=None, profile=None, use_ch=True):
    d = W.load_episode(f'{ROOT}/{r.path}'); asof, stale, info = W.observe(d['X'], profile or r.profile, int(r.seed) + 777, drop=drop)
    chron = tref = None
    if use_ch and USE_CH and os.path.exists(ch_path(r.split, r.ep_id)):
        z = np.load(ch_path(r.split, r.ep_id)); chron = z['feats'].astype(np.float32); tref = z['t_refresh']
    elif use_ch and USE_CH: return None
    F, names, groups = W.build_features(asof, stale, d['fc'], city, ORIG, hawkes=HK if USE_HK else None, chronos=chron, t_refresh=tref)
    on = d['onset'].astype(bool); typ = np.concatenate([on, on.any(-1, keepdims=True)], -1)                       # (T,Z,8)
    Y = W.window_labels(typ, ORIG, H)                                                                              # (n_o, NH, Z, 8)
    YC = W.window_labels(d['cas_onset'].astype(bool)[:, None], ORIG, H)[..., 0]                                    # (n_o, NH)
    nxt = np.where(on.any(-1), np.arange(on.shape[0])[:, None], 10**6); nxt = np.minimum.accumulate(nxt[::-1], 0)[::-1]
    nxt = np.vstack([nxt[1:], np.full((1, nxt.shape[1]), 10**6)]); tto = (nxt[ORIG] - ORIG[:, None]).astype(np.float32)   # hours to next 'any' onset after origin
    Xf = W.fill_defaults(asof); crit = W.critical_active_obs(Xf)[ORIG]
    return dict(F=F, names=names, groups=groups, Y=Y, YC=YC, tto=tto, crit=crit, onany=on.any(-1), hazard=r.hazard, profile=r.profile, ep=r.ep_id, trig=r.trigger_t, info=info)
def build_split(split, n=None, use_ch=True, **kw):
    rows = MAN[MAN.split == split]; out = []
    for r in tqdm(list(rows.itertuples()), desc=f'features {split}'):
        e = build_episode(r, CITY[r.city], use_ch=use_ch, **kw)
        if e is not None: out.append(e)
        if n and len(out) >= n: break
    return out
EPS = {s: build_split(s) for s in ['train', 'val', 'test', 'heldout', 'drift']}; NAMES, GROUPS = EPS['train'][0]['names'], np.array(EPS['train'][0]['groups'])
print({s: len(v) for s, v in EPS.items()}, '| n_features', len(NAMES), '| groups', pd.Series(GROUPS).value_counts().to_dict())
assert len(EPS['train']) > 0 and len(EPS['val']) > 1 and len(EPS['test']) > 0, 'not enough episodes (check chronos quota / NB00 output)'

In [ ]:
def pack(eps):
    n = len(eps); Z = eps[0]['F'].shape[1]; no = len(ORIG)
    d = dict(F=np.stack([e['F'] for e in eps]), Y=np.stack([e['Y'] for e in eps]), YC=np.stack([e['YC'] for e in eps]), tto=np.stack([e['tto'] for e in eps]), crit=np.stack([e['crit'] for e in eps]), n=n, Z=Z, no=no)
    d['Fr'] = d['F'].reshape(n * no * Z, -1); d['Yr'] = d['Y'].transpose(0, 1, 3, 2, 4).reshape(n * no * Z, NH, NT)            # row order (ep, origin, zone)
    d['t'] = np.tile(np.repeat(ORIG, Z), n); d['zone'] = np.tile(np.arange(Z), n * no); d['epi'] = np.repeat(np.arange(n), no * Z)
    return d
P = {s: pack(EPS[s]) for s in EPS}
# validation episodes split in two halves: early stopping vs calibration/thresholds (never the test set)
vi = np.arange(P['val']['n']); VES, VCAL = (vi % 2 == 0), (vi % 2 == 1)
sel = lambda d, m: dict(Fr=d['Fr'][m[d['epi']]], Yr=d['Yr'][m[d['epi']]], epi=d['epi'][m[d['epi']]], t=d['t'][m[d['epi']]], zone=d['zone'][m[d['epi']]])
RV_ES, RV_CAL = sel(P['val'], VES), sel(P['val'], VCAL)
base = P['train']['Yr'].mean(0); print('train base rates (rows x horizon x type) at h=24:', dict(zip(TYPES, base[:, H.index(24)].round(4))) if base.ndim == 2 else '')
CI = {n: i for i, n in enumerate(NAMES)}; print('rows train/val/test:', len(P['train']['Fr']), len(P['val']['Fr']), len(P['test']['Fr']))

## Training — one LightGBM per event type (horizon feature, monotone +1), checkpointed

In [ ]:
def mono(nf): return [0] * nf + [1]
def sample_rows(Fr, Yr, k, rng, n_max=None):
    N = len(Fr); idx = np.arange(N) if (n_max is None or N <= n_max) else rng.choice(N, n_max, replace=False); hi = rng.integers(0, NH, len(idx))
    X = np.hstack([Fr[idx], np.array(H, np.float32)[hi][:, None]]); return X, Yr[idx, hi, k].astype(int)
PARAMS = dict(objective='binary', num_leaves=63, learning_rate=0.05, min_data_in_leaf=200, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=7)
def train_booster(name, Ftr, Ytr, Fva, Yva, k, cols=None, rounds=None, n_max=None, tag=None):
    ck = f'{OUT}/model_{name}.txt'
    if os.path.exists(ck): return lgb.Booster(model_file=ck)
    rng = np.random.default_rng(k); c = np.arange(Ftr.shape[1]) if cols is None else cols
    Xtr, ytr = sample_rows(Ftr[:, c], Ytr, k, rng, n_max); Xva, yva = sample_rows(Fva[:, c], Yva, k, rng)
    pr = dict(PARAMS, monotone_constraints=mono(len(c))); pbar = tqdm(total=rounds or CFG['rounds'], desc=f'lgb {tag or name}', leave=False)
    cb = lambda env: pbar.update(1)
    b = lgb.train(pr, lgb.Dataset(Xtr, ytr), num_boost_round=rounds or CFG['rounds'], valid_sets=[lgb.Dataset(Xva, yva)], callbacks=[lgb.early_stopping(CFG['es'], verbose=False), lgb.log_evaluation(0), cb]); pbar.close(); b.save_model(ck); return b
def predict_h(b, Fr, cols=None, batch=400_000):
    c = np.arange(Fr.shape[1]) if cols is None else cols; out = np.zeros((len(Fr), NH), np.float32)
    for hi, h in enumerate(H):
        for s in range(0, len(Fr), batch): out[s:s + batch, hi] = b.predict(np.hstack([Fr[s:s + batch][:, c], np.full((len(Fr[s:s + batch]), 1), h, np.float32)]))
    return out
BOOST = {}
for k, n in enumerate(TYPES): BOOST[n] = train_booster(f'zone_{n}', P['train']['Fr'], P['train']['Yr'], RV_ES['Fr'], RV_ES['Yr'], k, tag=n)
# city-level cascade model: features = mean & max over zones of each zone feature
def city_rows(d): F = d['F']; return np.concatenate([F.mean(2), F.max(2)], -1).reshape(d['n'] * d['no'], -1)
CR = {s: city_rows(P[s]) for s in P}; YCr = {s: P[s]['YC'].reshape(-1, NH) for s in P}
vm = np.repeat(VES, P['val']['no']); ck_c = f'{OUT}/model_cascade.txt'
if os.path.exists(ck_c): BOOST['cascade'] = lgb.Booster(model_file=ck_c)
else:
    rng = np.random.default_rng(99); hi = rng.integers(0, NH, len(CR['train'])); hv = rng.integers(0, NH, vm.sum())
    pr = dict(PARAMS, min_data_in_leaf=50, num_leaves=15, monotone_constraints=mono(CR['train'].shape[1]))
    BOOST['cascade'] = lgb.train(pr, lgb.Dataset(np.hstack([CR['train'], np.array(H, np.float32)[hi][:, None]]), YCr['train'][np.arange(len(hi)), hi].astype(int)), num_boost_round=CFG['rounds'],
                                 valid_sets=[lgb.Dataset(np.hstack([CR['val'][vm], np.array(H, np.float32)[hv][:, None]]), YCr['val'][vm][np.arange(len(hv)), hv].astype(int))], callbacks=[lgb.early_stopping(CFG['es'], verbose=False), lgb.log_evaluation(0)]); BOOST['cascade'].save_model(ck_c)
print('trained:', list(BOOST))

## Calibration (isotonic per type × horizon on validation-cal episodes) and test metrics

In [ ]:
def raw_pred(split_rows_F, name): return predict_h(BOOST[name], split_rows_F)
RAW = {s: {n: raw_pred(P[s]['Fr'], f'{n}') for n in TYPES} for s in ['val', 'test']}
RAWC = {s: np.stack([BOOST['cascade'].predict(np.hstack([CR[s], np.full((len(CR[s]), 1), h, np.float32)])) for h in H], 1) for s in ['val', 'test']}
mcal = VCAL[P['val']['epi']]; mcal_c = np.repeat(VCAL, P['val']['no']); ISO = {}
for k, n in enumerate(TYPES):
    for hi in range(NH): ISO[(n, hi)] = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip').fit(RAW['val'][n][mcal, hi], P['val']['Yr'][mcal, hi, k])
for hi in range(NH): ISO[('cascade', hi)] = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip').fit(RAWC['val'][mcal_c, hi], YCr['val'][mcal_c, hi])
def calibrate(raw, n): p = np.stack([ISO[(n, hi)].predict(raw[:, hi]) for hi in range(NH)], 1); return np.maximum.accumulate(p, axis=1)       # monotone CDF in h
CAL = {s: {n: calibrate(RAW[s][n], n) for n in TYPES} for s in RAW}; CALC = {s: calibrate(RAWC[s], 'cascade') for s in RAW}
pickle.dump(ISO, open(f'{OUT}/isotonic.pkl', 'wb')); print('calibrators saved')
def ece_of(p, y, nb=10):
    o = np.argsort(p); bs = np.array_split(o, nb); return float(sum(len(b) * abs(p[b].mean() - y[b].mean()) for b in bs) / len(p))
def prob_metrics(p, y):
    if y.sum() == 0 or y.sum() == len(y): return dict(auroc=np.nan, ap=np.nan, brier=brier_score_loss(y, p), skill=np.nan, ece=ece_of(p, y), base=y.mean())
    br = brier_score_loss(y, p); br0 = y.mean() * (1 - y.mean()); return dict(auroc=roc_auc_score(y, p), ap=average_precision_score(y, p), brier=br, skill=1 - br / br0, ece=ece_of(p, y), base=y.mean())
rows = []
for k, n in enumerate(TYPES):
    for hi, h in enumerate(H): rows.append(dict(type=n, h=h, **prob_metrics(CAL['test'][n][:, hi], P['test']['Yr'][:, hi, k])))
for hi, h in enumerate(H): rows.append(dict(type='cascade(city)', h=h, **prob_metrics(CALC['test'][:, hi], YCr['test'][:, hi])))
PM_ = pd.DataFrame(rows); display(PM_[PM_.type.isin(['any', 'cascade(city)'])].round(4))

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(19, 8)); a = ax[0]
for j, h in enumerate([1, 6, 24, 72]):
    hi = H.index(h); p, y = CAL['test']['any'][:, hi], P['test']['Yr'][:, hi, TYPES.index('any')]; o = np.argsort(p); bs = np.array_split(o, 12)
    mp = [p[b].mean() for b in bs]; my = [y[b].mean() for b in bs]; a[j].plot([0, max(mp + my)], [0, max(mp + my)], 'k--'); a[j].plot(mp, my, 'o-'); a[j].set_title(f"reliability, 'any', h={h}  ECE={ece_of(p, y):.3f}"); a[j].set_xlabel('forecast'); a[j].set_ylabel('observed')
piv = PM_.pivot(index='type', columns='h', values='auroc'); im = ax[1, 0].imshow(piv.values, vmin=.5, vmax=1, cmap='viridis', aspect='auto'); ax[1, 0].grid(False); ax[1, 0].set_yticks(range(len(piv))); ax[1, 0].set_yticklabels(piv.index); ax[1, 0].set_xticks(range(NH)); ax[1, 0].set_xticklabels(H); ax[1, 0].set_title('AUROC type × horizon'); plt.colorbar(im, ax=ax[1, 0])
piv = PM_.pivot(index='type', columns='h', values='ece'); im = ax[1, 1].imshow(piv.values, cmap='magma_r', aspect='auto'); ax[1, 1].grid(False); ax[1, 1].set_yticks(range(len(piv))); ax[1, 1].set_yticklabels(piv.index); ax[1, 1].set_xticks(range(NH)); ax[1, 1].set_xticklabels(H); ax[1, 1].set_title('ECE type × horizon'); plt.colorbar(im, ax=ax[1, 1])
g = PM_[PM_.type == 'any']; ax[1, 2].plot(g.h, g.skill, 'o-', label='Brier skill'); ax[1, 2].plot(g.h, g.auroc, 's-', label='AUROC'); ax[1, 2].set_xscale('log'); ax[1, 2].legend(); ax[1, 2].set_title("'any' event: skill vs lead"); ax[1, 2].set_xlabel('h')
for k in range(len(TYPES)):
    n = TYPES[k]; y = P['test']['Yr'][:, H.index(24), k]
    if 0 < y.sum() < len(y):
        from sklearn.metrics import precision_recall_curve; pp, rr, _ = precision_recall_curve(y, CAL['test'][n][:, H.index(24)]); ax[1, 3].plot(rr, pp, label=n)
ax[1, 3].legend(fontsize=6); ax[1, 3].set_title('precision-recall, h=24'); savefig('40_calibration_and_skill')

## Time-to-critical: conditional CDF inversion → CQR + Mondrian(evidence coverage) + ACI

In [ ]:
GR = np.array([0] + H, float)
def ttc_quantiles(cal_any, qs=(0.1, 0.5, 0.9)):
    p72 = cal_any[:, -1]; cdf = np.hstack([np.zeros((len(cal_any), 1)), cal_any]) / np.maximum(p72, 1e-6)[:, None]
    out = np.stack([np.array([np.interp(q, c, GR) for c in cdf]) for q in qs], 1); return out, p72         # hours, conditional on an event within 72 h
def ttc_set(split, pmin=0.3):
    d = P[split]; tto = d['tto'].reshape(-1); q, p72 = ttc_quantiles(CAL[split]['any']); ok = (tto <= 72) & (p72 >= pmin); cov_ = d['Fr'][:, CI['dq_cov']]; bn = np.where(cov_ >= .8, 0, np.where(cov_ >= .5, 1, 2)); return dict(q=q, p72=p72, tau=tto, ok=ok, bin=bn, epi=d['epi'], t=d['t'])
TV, TT = ttc_set('val'), ttc_set('test'); mv = VCAL[TV['epi']] & TV['ok']; ALPHA = 0.2
scores = {b: np.maximum(TV['q'][mv & (TV['bin'] == b), 0] - TV['tau'][mv & (TV['bin'] == b)], TV['tau'][mv & (TV['bin'] == b)] - TV['q'][mv & (TV['bin'] == b), 2]) for b in range(3)}
def adj_for(b, level): s = scores[b] if len(scores[b]) > 10 else np.concatenate(list(scores.values())); return float(np.quantile(s, min(1, level * (1 + 1 / len(s)))))
te = TT['ok']; q = TT['q'][te]; tau = TT['tau'][te]; bn = TT['bin'][te]; tk = (TT['t'][te] + tau)
raw_hit = (tau >= q[:, 0]) & (tau <= q[:, 2]); adj = np.array([adj_for(b, 1 - ALPHA) for b in bn]); cqr_hit = (tau >= q[:, 0] - adj) & (tau <= q[:, 2] + adj)
order = np.argsort(tk); al = ALPHA; aci_hit = np.zeros(len(tau), bool); traj = []
for i in order:                                           # outcome of a row becomes known at t + tau: feed back in that order (ACI, gamma=0.005)
    a_ = adj_for(bn[i], float(np.clip(1 - al, 0.5, 0.995))); hit = (tau[i] >= q[i, 0] - a_) and (tau[i] <= q[i, 2] + a_); aci_hit[i] = hit; al = al + 0.005 * (ALPHA - (0 if hit else 1)); traj.append(al)
TTC = pd.DataFrame(dict(method=['raw CDF inversion', 'CQR + Mondrian', 'CQR + Mondrian + ACI'], coverage=[raw_hit.mean(), cqr_hit.mean(), aci_hit.mean()], nominal=0.8))
widths = [(q[:, 2] - q[:, 0]).mean(), (q[:, 2] - q[:, 0] + 2 * adj).mean(), np.nan]; TTC['mean_width_h'] = widths; TTC['n'] = len(tau); display(TTC.round(3))
byb = pd.DataFrame(dict(bin=['cov>=0.8', '0.5-0.8', '<0.5'])); byb['n'] = [int((bn == b).sum()) for b in range(3)]
for nm, h_ in [('raw', raw_hit), ('cqr', cqr_hit), ('aci', aci_hit)]: byb[nm] = [h_[bn == b].mean() if (bn == b).any() else np.nan for b in range(3)]
display(byb.round(3)); print(f'median |q50 - true time-to-critical| = {np.median(np.abs(q[:, 1] - tau)):.1f} h on {len(tau)} events (rows with an onset within 72 h and P(72h)>=0.3)')
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); ax[0].scatter(tau, q[:, 1], s=4, alpha=.3); ax[0].plot([0, 72], [0, 72], 'r--'); ax[0].set_xlabel('true hours to critical'); ax[0].set_ylabel('predicted median'); ax[0].set_title('time-to-critical')
x = np.arange(3); [ax[1].bar(x + o_, byb[c], .27, label=c) for o_, c in zip([-.27, 0, .27], ['raw', 'cqr', 'aci'])]; ax[1].axhline(.8, c='k', ls='--'); ax[1].set_xticks(x); ax[1].set_xticklabels(byb.bin); ax[1].set_title('80% interval coverage by evidence-coverage bin (Mondrian)'); ax[1].legend()
ax[2].plot(traj); ax[2].axhline(ALPHA, c='k', ls='--'); ax[2].set_title('ACI alpha_t trajectory'); savefig('41_time_to_critical')

## Risk states, alert episodes, **lead time at a fixed false-alarm budget**, baselines

In [ ]:
def alert_eval(score, eps, thr, tick=3, horizon=24):
    # score (n_ep, n_o, Z); returns dict(det, n_pos, leads, false_eps, zone_weeks)
    leads, n_pos, det, fa, zw = [], 0, 0, 0, 0.0
    for i, e in enumerate(eps):
        on = e['onany']; span = (ORIG[-1] - ORIG[0] + tick) / 168.0
        for z in range(score.shape[2]):
            a = score[i, :, z] >= thr; on_t = np.where(on[:, z])[0]; on_t = on_t[on_t > ORIG[0]]; zw += span
            idx = np.where(a)[0]; eps_ = []
            if len(idx):
                st = idx[0]; pv = idx[0]
                for j in idx[1:]:
                    if j - pv >= 3: eps_.append((st, pv)); st = j
                    pv = j
                eps_.append((st, pv))
            for (s_, e_) in eps_:
                s_t, e_t = ORIG[s_], ORIG[e_]
                if not ((on_t > s_t) & (on_t <= e_t + horizon)).any(): fa += 1
            if len(on_t):
                f = on_t[0]; n_pos += 1; cand = [ORIG[s_] for (s_, e_) in eps_ if ORIG[s_] < f and ORIG[e_] + horizon >= f]
                if cand: det += 1; leads.append(f - min(cand))
    return dict(det=det, n_pos=n_pos, leads=leads, false_eps=fa, zone_weeks=zw, far=fa / max(zw, 1e-9), rate=det / max(n_pos, 1), lead_med=float(np.median(leads)) if leads else np.nan)
def to_cube(arr, d): return arr.reshape(d['n'], d['no'], d['Z'])
def sweep(score_cube, eps, nthr=25):
    ts = np.unique(np.quantile(score_cube, np.linspace(0.5, 0.9995, nthr))); return pd.DataFrame([dict(thr=t, **{k: v for k, v in alert_eval(score_cube, eps, t).items() if k != 'leads'}) for t in ts])
def tune(sw, budget): ok = sw[sw.far <= budget]; return float(ok.thr.min()) if len(ok) else float(sw.thr.max())
hi24 = H.index(24); VC_eps = [e for e, m in zip(EPS['val'], VCAL) if m]
sc_val = to_cube(CAL['val']['any'][:, hi24], P['val'])[VCAL]; sc_te = to_cube(CAL['test']['any'][:, hi24], P['test'])
SW_val = sweep(sc_val, VC_eps); THR_H = tune(SW_val, CFG['far_budget']); SW_te = sweep(sc_te, EPS['test']); res = alert_eval(sc_te, EPS['test'], THR_H)
print(f'HIGH-alert threshold tau_H={THR_H:.3f} (tuned on val-cal for <= {CFG["far_budget"]} false alert-episodes / zone-week)')
print(f'TEST: detection {res["det"]}/{res["n_pos"]} = {res["rate"]:.2f} | FAR {res["far"]:.2f}/zone-week | lead median {res["lead_med"]:.1f} h, IQR {np.percentile(res["leads"], [25, 75]) if res["leads"] else None}')

In [ ]:
# baselines on the same alert logic: persistence/threshold rule on current worst critical-layer stress; Chronos-only thresholding
sidx = [CI[f's_{l}'] for l in ['TRF', 'PWR', 'HOS', 'AIR', 'SEI', 'COM']]
def persist_score(d, Fr): return to_cube(Fr[:, sidx].max(1), d)
BASE = {'GBM + cascade features (ours)': (sc_val, sc_te), 'persistence / threshold rule': (persist_score(P['val'], P['val']['Fr'])[VCAL], persist_score(P['test'], P['test']['Fr']))}
if USE_CH:
    def ch_score(d):
        F = d['Fr']; g = lambda n: F[:, CI[f'ch_{n}_h24_q90']]
        s = np.stack([g('water_depth_cm') / 25, (1 - F[:, CI['ch_speed_idx_h24_q10']]) / .8, (g('load_ratio') - .75) / .4, (g('bed_occ') - .5) / .45, (g('ed_ratio') - 1) / 3, (g('wbgt') - 26) / 8], 1); return to_cube(s.max(1), d)
    BASE['Chronos-2 only + threshold'] = (ch_score(P['val'])[VCAL], ch_score(P['test']))
CURVES = {}
for nm, (sv, st) in BASE.items():
    swv = sweep(sv, VC_eps); thr = tune(swv, CFG['far_budget']); r_ = alert_eval(st, EPS['test'], thr); CURVES[nm] = sweep(st, EPS['test']); print(f'{nm:34s} FAR {r_["far"]:.2f} detect {r_["rate"]:.2f} lead median {r_["lead_med"]:.1f} h')
fig, ax = plt.subplots(1, 3, figsize=(18, 4.4))
for nm, c in CURVES.items(): c = c.sort_values('far'); ax[0].plot(c.far, c.lead_med, 'o-', label=nm); ax[1].plot(c.far, c.rate, 'o-', label=nm)
for a, t in zip(ax[:2], ['median lead time (h)', 'detection rate']): a.axvline(CFG['far_budget'], c='k', ls='--'); a.set_xlabel('false alert-episodes per zone-week'); a.set_ylabel(t); a.set_xlim(0, 4); a.legend(fontsize=7)
ax[0].set_title('LEAD TIME vs FALSE-ALARM RATE (test)'); ax[1].set_title('detection vs FALSE-ALARM RATE'); ax[2].hist(res['leads'], bins=20, color='tab:green'); ax[2].set_title(f'lead-time distribution at FAR budget (n={len(res["leads"])})'); ax[2].set_xlabel('hours'); savefig('42_lead_time_vs_far')

In [ ]:
# 4-state mapping with hysteresis (zone) + city state (cascade model); thresholds scaled from the FAR-tuned tau_H
TAU = dict(W=0.5 * THR_H, H=THR_H, C=min(0.95, 1.6 * THR_H)); print('state thresholds', {k: round(v, 3) for k, v in TAU.items()})
def risk_states(p12, p24, p72, crit, tau=TAU):                       # arrays (n_ep, n_o, Z); returns int 0..3 with hysteresis (upgrade now, downgrade after 2 lower ticks)
    raw = np.where(crit | (p12 >= tau['C']), 3, np.where(p24 >= tau['H'], 2, np.where(p72 >= tau['W'], 1, 0))); out = raw.copy()
    for i in range(raw.shape[0]):
        for z in range(raw.shape[2]):
            cur, low = 0, 0
            for j in range(raw.shape[1]):
                r_ = raw[i, j, z]
                if r_ >= cur: cur, low = r_, 0
                else:
                    low += 1
                    if low >= 2: cur, low = r_, 0
                out[i, j, z] = cur
    return out
def zone_cube(split, hz): return to_cube(CAL[split]['any'][:, H.index(hz)], P[split])
ST_te = risk_states(zone_cube('test', 12), zone_cube('test', 24), zone_cube('test', 72), P['test']['crit']); names_st = ['NORMAL', 'WATCH', 'HIGH RISK', 'CRITICAL']
pc = CALC['test'].reshape(P['test']['n'], P['test']['no'], NH); city_state = np.where(pc[..., H.index(24)] >= TAU['C'], 3, np.where(pc[..., H.index(48)] >= TAU['H'], 2, np.where(pc[..., -1] >= TAU['W'], 1, 0)))
hz_ = np.array([e['hazard'] for e in EPS['test']]); dist = pd.DataFrame({h: np.bincount(ST_te[hz_ == h].ravel(), minlength=4) / max(ST_te[hz_ == h].size, 1) for h in W.HAZARDS if (hz_ == h).any()}, index=names_st).T; display(dist.round(3))
# first-WATCH / first-HIGH lead before first critical onset, per hazard
rows = []
for i, e in enumerate(EPS['test']):
    for z in range(P['test']['Z']):
        on = np.where(e['onany'][:, z])[0]; on = on[on > ORIG[0]]
        if not len(on): continue
        f = on[0]
        for lvl, nm in [(1, 'WATCH'), (2, 'HIGH RISK')]:
            j = np.where((ST_te[i, :, z] >= lvl) & (ORIG < f))[0]; rows.append(dict(hazard=e['hazard'], level=nm, lead=(f - ORIG[j[0]]) if len(j) else np.nan))
LD = pd.DataFrame(rows); print('median lead (h) of first WATCH / HIGH before first critical onset, by hazard (NaN share = missed):'); display(LD.groupby(['hazard', 'level']).lead.agg(['median', lambda s: s.isna().mean()]).unstack().round(1))
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6)); dist.plot(kind='barh', stacked=True, ax=ax[0], color=['#2ca02c', '#f2c744', '#ff7f0e', '#d62728']); ax[0].set_title('time spent in each state, by hazard (test)'); ax[0].legend(fontsize=7)
LDp = LD.dropna(); [ax[1].boxplot([LDp[(LDp.level == lv) & (LDp.hazard == h)].lead for h in sorted(LDp.hazard.unique())], positions=np.arange(len(LDp.hazard.unique())) + o_, widths=.3) for lv, o_ in [('WATCH', -.17), ('HIGH RISK', .17)]]
ax[1].set_xticks(range(len(LDp.hazard.unique()))); ax[1].set_xticklabels(sorted(LDp.hazard.unique()), rotation=30); ax[1].set_ylabel('hours before first critical'); ax[1].set_title('lead of WATCH (left) and HIGH (right) boxes')
i0 = next((i for i, e in enumerate(EPS['test']) if e['hazard'] == 'flood'), 0); im = ax[2].imshow(ST_te[i0].T, aspect='auto', cmap=matplotlib.colors.ListedColormap(['#2ca02c', '#f2c744', '#ff7f0e', '#d62728']), vmin=0, vmax=3, extent=[ORIG[0], ORIG[-1], P['test']['Z'], 0]); ax[2].grid(False)
ax[2].set_title(f"{EPS['test'][i0]['ep']} zone states over time (trigger t={EPS['test'][i0]['trig']})"); ax[2].axvline(EPS['test'][i0]['trig'], c='k', ls='--'); ax[2].set_xlabel('hour'); ax[2].set_ylabel('zone'); savefig('43_risk_states')

## Ablations (same episodes, same budget) — what each family of features buys

In [ ]:
kA = TYPES.index('any'); ABL = {'full': None, '- hawkes': ['hawkes'], '- chronos': ['chronos'], '- forecast covariates': ['forecast_cov'], '- early-warning signals': ['ews'], '- data-quality': ['dq'], '- neighbours/city': ['neighbor'], '- trigger ages': ['trigger']}
if not USE_HK: ABL.pop('- hawkes')
if not USE_CH: ABL.pop('- chronos')
rows = []; MODELS_ABL = {}
for nm, drop in ABL.items():
    cols = np.where(~np.isin(GROUPS, drop or []))[0]; key = nm.replace(' ', '').replace('-', 'no_').replace('/', '_')
    b = train_booster(f'abl_{key}', P['train']['Fr'], P['train']['Yr'], RV_ES['Fr'], RV_ES['Yr'], kA, cols=cols, n_max=CFG['abl_rows'], tag=nm, rounds=min(CFG['rounds'], 300)); MODELS_ABL[nm] = (b, cols)
    rv = predict_h(b, P['val']['Fr'], cols); rt = predict_h(b, P['test']['Fr'], cols); iso = [IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip').fit(rv[mcal, hi], P['val']['Yr'][mcal, hi, kA]) for hi in range(NH)]
    pt = np.maximum.accumulate(np.stack([iso[hi].predict(rt[:, hi]) for hi in range(NH)], 1), axis=1); pv = np.maximum.accumulate(np.stack([iso[hi].predict(rv[:, hi]) for hi in range(NH)], 1), axis=1)
    swv = sweep(to_cube(pv[:, hi24], P['val'])[VCAL], VC_eps, 15); thr = tune(swv, CFG['far_budget']); r_ = alert_eval(to_cube(pt[:, hi24], P['test']), EPS['test'], thr); m24 = prob_metrics(pt[:, hi24], P['test']['Yr'][:, hi24, kA]); m6 = prob_metrics(pt[:, H.index(6)], P['test']['Yr'][:, H.index(6), kA])
    rows.append(dict(variant=nm, n_features=len(cols), AUROC_24h=m24['auroc'], AP_24h=m24['ap'], Brier_skill_24h=m24['skill'], ECE_24h=m24['ece'], AUROC_6h=m6['auroc'], detect=r_['rate'], FAR=r_['far'], lead_med_h=r_['lead_med']))
ABLT = pd.DataFrame(rows); display(ABLT.round(3)); ABLT.to_csv(f'{OUT}/ablations.csv', index=False)
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); [ax[i].barh(ABLT.variant[::-1], ABLT[c][::-1], color=col) for i, (c, col) in enumerate([('AUROC_24h', 'tab:blue'), ('Brier_skill_24h', 'tab:green'), ('lead_med_h', 'tab:red')])]
[ax[i].set_title(t) for i, t in enumerate(['AUROC @24h', 'Brier skill @24h', 'median lead (h) at FAR budget'])]; savefig('44_ablations')

## Adversity: fault profiles, and the dropout sweep (graceful degradation)

In [ ]:
rows = []
for pf in W.PROFILE_NAMES:
    m = np.array([e['profile'] == pf for e in EPS['test']])
    if m.sum() < 1: continue
    mr = m[P['test']['epi']]; y = P['test']['Yr'][mr, hi24, kA]; p = CAL['test']['any'][mr, hi24]; sub_eps = [e for e, k in zip(EPS['test'], m) if k]; r_ = alert_eval(to_cube(p, dict(n=m.sum(), no=P['test']['no'], Z=P['test']['Z'])), sub_eps, THR_H)
    rows.append(dict(profile=pf, n_eps=int(m.sum()), AUROC_24h=prob_metrics(p, y)['auroc'], ECE_24h=ece_of(p, y), Brier_skill=prob_metrics(p, y)['skill'], detect=r_['rate'], FAR=r_['far'], lead_med_h=r_['lead_med'], mean_evidence_cov=float(P['test']['Fr'][mr][:, CI['dq_cov']].mean())))
PROF = pd.DataFrame(rows); display(PROF.round(3))
# dropout sweep with the chronos-free model (Chronos features exist only for the assigned profile) ; same tuned threshold
nm_ = '- chronos' if '- chronos' in MODELS_ABL else 'full'; b_, cols_ = MODELS_ABL[nm_]; rv = predict_h(b_, P['val']['Fr'], cols_); iso = [IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip').fit(rv[mcal, hi], P['val']['Yr'][mcal, hi, kA]) for hi in range(NH)]
pv = np.maximum.accumulate(np.stack([iso[hi].predict(rv[:, hi]) for hi in range(NH)], 1), axis=1); thr0 = tune(sweep(to_cube(pv[:, hi24], P['val'])[VCAL], VC_eps, 15), CFG['far_budget'])
sub = MAN[MAN.ep_id.isin([e['ep'] for e in EPS['test'][:CFG['n_sweep_eps']]])]; SWP = []
for dr in tqdm([0, .1, .2, .3, .4, .5], desc='dropout sweep'):
    es = [build_episode(r, CITY[r.city], drop=dr, profile='F0') for r in sub.itertuples()]; d = pack(es); rt = predict_h(b_, d['Fr'], cols_)
    pt = np.maximum.accumulate(np.stack([iso[hi].predict(rt[:, hi]) for hi in range(NH)], 1), axis=1); y = d['Yr'][:, hi24, kA]; r_ = alert_eval(to_cube(pt[:, hi24], d), es, thr0)
    SWP.append(dict(dropout=dr, AUROC=prob_metrics(pt[:, hi24], y)['auroc'], ECE=ece_of(pt[:, hi24], y), detect=r_['rate'], FAR=r_['far'], lead_med=r_['lead_med'], cov=float(d['Fr'][:, CI['dq_cov']].mean())))
SWP = pd.DataFrame(SWP); SWP['lead_retention'] = SWP.lead_med / SWP.lead_med.iloc[0]; display(SWP.round(3))
fig, ax = plt.subplots(1, 4, figsize=(19, 3.8)); [ax[i].plot(SWP.dropout * 100, SWP[c], 'o-') for i, c in enumerate(['lead_retention', 'ECE', 'AUROC', 'FAR'])]
[ax[i].set_title(t) for i, t in enumerate(['lead-time retention (target >=0.7 @30%)', 'ECE vs % sources missing', 'AUROC', 'false alerts / zone-week'])]; [a.set_xlabel('% readings dropped') for a in ax]; ax[0].axhline(.7, c='r', ls='--'); savefig('45_dropout_sweep')

## Held-out city (never tuned on) + explainability

In [ ]:
hv_ = P['heldout']; ph = {n: calibrate(predict_h(BOOST[n], hv_['Fr']), n) for n in ['any']}; y = hv_['Yr'][:, hi24, kA]; mh = prob_metrics(ph['any'][:, hi24], y); rh = alert_eval(to_cube(ph['any'][:, hi24], hv_), EPS['heldout'], THR_H); ma = prob_metrics(CAL['test']['any'][:, hi24], P['test']['Yr'][:, hi24, kA])
HO = pd.DataFrame([dict(set='anchor-test', **{k: ma[k] for k in ('auroc', 'ap', 'skill', 'ece')}, detect=res['rate'], FAR=res['far'], lead_med=res['lead_med']),
                   dict(set='HELD-OUT city', **{k: mh[k] for k in ('auroc', 'ap', 'skill', 'ece')}, detect=rh['rate'], FAR=rh['far'], lead_med=rh['lead_med'])]); display(HO.round(3))
print('relative drop in AUROC: %.1f%%  (target <= 25%%)  | lead retention %.2f' % (100 * (ma['auroc'] - mh['auroc']) / ma['auroc'], rh['lead_med'] / max(res['lead_med'], 1e-9)))
# grouped TreeSHAP on the 'any' model at h=24
rng = np.random.default_rng(0); ix = rng.choice(len(P['test']['Fr']), min(CFG['shap_rows'], len(P['test']['Fr'])), replace=False); Xs = np.hstack([P['test']['Fr'][ix], np.full((len(ix), 1), 24, np.float32)])
con = BOOST['any'].predict(Xs, pred_contrib=True)[:, :-1]; GN = list(pd.unique(GROUPS)) + ['horizon']; gl = np.array(list(GROUPS) + ['horizon']); G_ = {g: np.abs(con[:, gl == g]).sum(1) for g in GN}
imp = pd.Series({g: v.mean() for g, v in G_.items()}).sort_values(); fig, ax = plt.subplots(1, 2, figsize=(14, 4.4)); imp.plot.barh(ax=ax[0], color='tab:purple'); ax[0].set_title("mean |SHAP| by feature group ('any', h=24)")
yv = P['test']['Yr'][ix, hi24, kA]; sg = pd.DataFrame({g: np.abs(con[:, gl == g]).sum(1) for g in GN}); sg['pos'] = yv; sg.groupby('pos')[GN].mean().T.plot.barh(ax=ax[1]); ax[1].set_title('driver mix: no-event rows (0) vs event rows (1)'); savefig('46_shap_groups')
fi = pd.Series(BOOST['any'].feature_importance('gain'), index=NAMES + ['horizon']).sort_values(ascending=False).head(15); print('top features by gain:'); print((fi / fi.sum()).round(3).to_string())

In [ ]:
# ---------------- save artifacts (+ prediction logs for NB05)
json.dump(dict(names=NAMES, groups=list(GROUPS), tau=TAU, far_budget=CFG['far_budget'], thr_high=THR_H, orig=ORIG.tolist(), horizons=H, use_chronos=USE_CH, use_hawkes=USE_HK, types=TYPES), open(f'{OUT}/risk_config.json', 'w'), indent=1)
def dump(split, name):
    d = P[split]; eps = EPS[split]; raw = RAW[split]['any'] if split in RAW else predict_h(BOOST['any'], d['Fr']); p = CAL[split]['any'] if split in CAL else calibrate(raw, 'any'); rows = []
    ep_id = np.array([e['ep'] for e in eps]); prof = np.array([e['profile'] for e in eps]); hz = np.array([e['hazard'] for e in eps]); cov = d['Fr'][:, CI['dq_cov']]
    for hi, h in enumerate(H):
        rows.append(pd.DataFrame(dict(ep=ep_id[d['epi']], profile=prof[d['epi']], hazard=hz[d['epi']], t=d['t'], zone=d['zone'], h=h, logit_raw=np.log(np.clip(raw[:, hi], 1e-6, 1 - 1e-6) / (1 - np.clip(raw[:, hi], 1e-6, 1 - 1e-6))), p_cal=p[:, hi], y=d['Yr'][:, hi, kA].astype(int), cov=cov)))
    pd.concat(rows).to_csv(f'{OUT}/risk_predictions_{split}.csv.gz', index=False)
for s in ['test', 'drift', 'heldout']: dump(s, s)
for t_, n_ in [(PM_, 'metrics_by_type_horizon'), (PROF, 'metrics_by_fault_profile'), (SWP, 'dropout_sweep'), (HO, 'heldout_vs_anchor'), (TTC, 'time_to_critical_coverage')]: t_.to_csv(f'{OUT}/{n_}.csv', index=False)
print('saved: model_*.txt, isotonic.pkl, risk_config.json, risk_predictions_{test,drift,heldout}.csv.gz, metrics csvs')